# 🏥 MedRAX Reproduction: Safe Single-Account `gemini-3.5-flash-lite` Evaluation
### Reproducing "MedRAX: Medical Reasoning Agent for Chest X-ray" (ICML 2025)
This notebook evaluates **ChestAgentBench** using **`gemini-3.5-flash-lite`** with **Anti-Abuse Rate Pacing (Random Jitter)** and **Zero-Retry Quota Stop**:

| Feature | Design & Anti-Abuse Protection |
| :--- | :--- |
| **Model** | **`gemini-3.5-flash-lite`** (Multimodal Vision + Native Tool Calling) |
| **Zero-Retry on 429** | **`max_retries=0`** in LangChain + **0 retries on `429`** in script: The instant a quota limit is reached, traffic stops 100% (never repeats rejected requests!) |
| **Human-Like Jitter Pacer** | **Randomized Interval (`5.6s – 6.6s` = ~9.5–10 RPM)**: Safely 35% below the 15 RPM limit with organic random jitter so it never looks like a fixed-timer bot |
| **Session Question Cap** | Supports `--max-questions N` so you can stop cleanly after `N` questions *before* even hitting a single `429` |
| **Drive Sync & Resume** | **Saves every 10 questions (`os.fsync`)** + emergency flush on stop. Resume anytime without losing a single question! |


## 1. ⚙️ Hardware & GPU Environment Diagnostics
Verify the Colab runtime environment (CUDA availability, GPU model, and VRAM).
Since `gemini-3.5-flash-lite` runs on the cloud, 100% of your Colab GPU (15GB T4 or 22.5GB L4) is dedicated to MedRAX's radiology tools.


In [ ]:
# Check GPU availability and specifications
import os
import sys
import torch

print(f"Python Version : {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Detected GPU   : {device_name} ({vram_gb:.2f} GB VRAM)")
    print("🚀 Local GPU is 100% dedicated to MedRAX clinical vision tools!")
else:
    print("⚠️ WARNING: No GPU detected! Please enable GPU in Runtime -> Change runtime type.")

!nvidia-smi


## 2. 💾 Google Drive Mounting & Persistent Storage
Mount Google Drive to persist downloaded datasets, model checkpoints, and evaluation results (`predictions.jsonl`).
Every 10 questions (plus any remaining questions when an account hits its daily limit) are force-synced directly to Google Drive via `os.fsync`.


In [ ]:
# Mount Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/MedRAX_Thesis')

PERSISTENT_DIRS = {
    "checkpoints": DRIVE_ROOT / "checkpoints",
    "data": DRIVE_ROOT / "data" / "chestagentbench",
    "results": DRIVE_ROOT / "results",
    "logs": DRIVE_ROOT / "logs",
}

for name, path in PERSISTENT_DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"📁 Verified persistent directory: {path}")

# Set environment variables
os.environ["MEDRAX_MODEL_DIR"] = str(PERSISTENT_DIRS["checkpoints"])
os.environ["BENCHMARK_DATA_DIR"] = str(PERSISTENT_DIRS["data"])
os.environ["MEDRAX_RESULTS_DIR"] = str(PERSISTENT_DIRS["results"])
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

print("\n✅ Environment variables configured successfully.")


## 3. 🐙 Clone Reproduction Repository from GitHub
Clone the reproduction suite from GitHub. If already present, pull the latest code updates.


In [ ]:
# Configure GitHub repository
import os
from pathlib import Path

REPO_URL = "https://github.com/Uyhuynh23/MedRAX-ChestAgentBench.git"
LOCAL_REPO_DIR = Path("/content/MedRAX-ChestAgentBench")

if not LOCAL_REPO_DIR.exists():
    print(f"Cloning {REPO_URL} into {LOCAL_REPO_DIR} ...")
    !git clone {REPO_URL} {LOCAL_REPO_DIR}
else:
    print(f"Repository exists. Pulling latest updates...")
    %cd {LOCAL_REPO_DIR}
    !git pull origin main

%cd {LOCAL_REPO_DIR}
!python scripts/setup_drive.py


## 4. 📦 Install Dependencies & Specialized Radiology Packages
Install PyTorch, LangChain, TorchXRayVision, Transformers, and Google Generative AI SDK.


In [ ]:
%cd /content/MedRAX-ChestAgentBench

# Install project dependencies
!pip install -q -r requirements.txt
!pip install -q torchxrayvision
!pip install -q -U langchain-google-genai google-generativeai

print("\n✅ All dependencies installed successfully.")


## 5. 🔑 Configure Active Account API Key (1 Key & 1 IP Session at a time)

### 🛡️ Golden Rules so Google NEVER flags your account for abuse:
1. **Only 1 API Key in this session**: Do **NOT** store multiple accounts' keys in the same Colab notebook.
2. **Zero-Retry on 429**: The code is configured with `max_retries=0` and **0 retries on `429`**. The exact moment a quota limit is reached, it saves to Google Drive and stops 100% of API requests immediately (never repeating rejected requests).
3. **When switching to another Google Account**:
   * **IMPORTANT**: Do **NOT** simply paste Account #2's key into the same running Colab VM right after Account #1 hits its limit (because the running Colab VM has the same IP address!).
   * Instead, click **`Runtime -> Disconnect and delete runtime`** (Ngắt kết nối và xóa thời gian chạy) so Google Colab assigns a **brand-new VM with a new IP address** (or open the notebook in Account #2's own browser profile), and then resume from Google Drive!


In [ ]:
# -------------------------------------------------------------
# 🔑 Set Active Gemini API Key (Single Account Only)
# -------------------------------------------------------------
import os
from getpass import getpass

active_key = None
try:
    from google.colab import userdata
    for candidate_secret in ["GEMINI_API_KEY", "GOOGLE_API_KEY"]:
        try:
            val = userdata.get(candidate_secret)
            if val and val.strip():
                active_key = val.split(",")[0].strip()
                print(f"🔑 Loaded key from Colab Secret [{candidate_secret}]")
                break
        except Exception:
            pass
except Exception:
    pass

if not active_key:
    active_key = getpass("Enter your single active GEMINI_API_KEY: ").strip()

if active_key:
    os.environ["GEMINI_API_KEY"] = active_key
    os.environ["GOOGLE_API_KEY"] = active_key
    masked = f"{active_key[:6]}...{active_key[-6:]}" if len(active_key) > 12 else "***"
    print(f"✅ Active API Key configured: {masked} (Zero-Retry Anti-Abuse Mode)")
else:
    print("❌ No API key provided!")


In [ ]:
# -------------------------------------------------------------
# 🧪 Single-Key Vision Ground-Truth Proof & Latency Check
# -------------------------------------------------------------
# Draws secret code "7492" inside image pixels (never in text prompt)
# and verifies that gemini-3.5-flash-lite reads it accurately!
import time
import os
import google.generativeai as genai
from PIL import Image, ImageDraw

active_key = os.environ.get("GEMINI_API_KEY", "").strip()
MODEL_NAME = "gemini-3.5-flash-lite"

# Draw secret visual code "7492" onto image canvas
test_img = Image.new("RGB", (320, 120), color=(20, 20, 20))
draw = ImageDraw.Draw(test_img)
draw.rectangle([10, 10, 310, 110], outline=(220, 220, 220), width=3)
draw.text((40, 45), "SECRET CODE: 7492", fill=(255, 255, 255))

print("=======================================================")
print(f" Running Single-Key Vision & Latency Check ({MODEL_NAME})")
print("=======================================================\n")

try:
    genai.configure(api_key=active_key)
    model = genai.GenerativeModel(MODEL_NAME)
    t0 = time.perf_counter()
    resp = model.generate_content([
        "Read the exact 4-digit secret code written inside this image.",
        test_img
    ])
    elapsed = time.perf_counter() - t0
    ans = " ".join((resp.text or "").split())
    if "7492" in ans:
        print(f"  ✅ VISION PROVEN ({elapsed:.2f}s | {MODEL_NAME}) -> Read '7492' from image! Output: {ans}")
    else:
        print(f"  ⚠️ Response received ({elapsed:.2f}s): {ans}")
except Exception as e:
    print(f"  ❌ FAILED -> {e}")


## 6. 📥 Download ChestAgentBench Dataset & Extract Figures
Download the **ChestAgentBench** benchmark dataset (`wanglab/chest-agent-bench`) from Hugging Face into Google Drive.
If already present on Google Drive, it will verify integrity and skip re-downloading.


In [ ]:
# Download / Verify ChestAgentBench in persistent Google Drive
%cd /content/MedRAX-ChestAgentBench
!python scripts/download_benchmark.py --data-dir "$BENCHMARK_DATA_DIR"

!ls -lh "$BENCHMARK_DATA_DIR"
!head -n 2 "$BENCHMARK_DATA_DIR/metadata.jsonl" | cut -c 1-120


## 7. 🔍 Check Saved Progress on Google Drive
Run the cell below anytime to check how many questions are already saved in Google Drive before starting or resuming a run.


In [ ]:
# -------------------------------------------------------------
# 📊 Check Saved Progress on Google Drive
# -------------------------------------------------------------
import json
from pathlib import Path

results_dir = Path(os.environ["MEDRAX_RESULTS_DIR"])
for run_name in ["gemini_baseline_full", "gemini_medrax_full"]:
    pred_file = results_dir / run_name / "predictions.jsonl"
    if pred_file.exists():
        lines = [json.loads(line) for line in open(pred_file, encoding="utf-8") if line.strip()]
        corr = sum(1 for r in lines if r.get("is_correct"))
        acc = (corr / len(lines) * 100) if lines else 0.0
        print(f"📁 {run_name:<22}: {len(lines):>4}/2500 questions saved | Running Acc: {acc:.2f}% ({corr}/{len(lines)})")
    else:
        print(f"📁 {run_name:<22}:    0/2500 questions saved (Not started yet)")


## 8. 🧪 Benchmark Execution (`gemini-3.5-flash-lite` + Safe Auto-Resume)

### How Account Handover Works:
1. Run **Option A (Baseline)** or **Option B (MedRAX Agent)** below.
2. Every **10 questions**, results are force-synced (`os.fsync`) to Google Drive.
3. When your current account hits its daily quota limit, the script **immediately flushes any remaining 1–9 questions to Google Drive and stops safely** (`🛑 [DAILY QUOTA REACHED - SAFE STOP]`).
4. To continue with your next account:
   * Go back to **Section 5 (Cell 10)**, switch to your next API key (`GEMINI_API_KEY_2` or paste the new key), and run Cell 10.
   * Come back here and **re-run the exact same cell** — because `--reset` is NOT passed, it will automatically print `[Resume] Found N questions already evaluated. Skipping them.` and continue seamlessly from question `N+1`!


In [ ]:
# -------------------------------------------------------------
# 🧪 Step 8.1a: Smoke Test - Baseline Mode on 1 Case (~15 sec)
# -------------------------------------------------------------
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py \
    --mode baseline \
    --model gemini-3.5-flash-lite \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/smoke_test_baseline" \
    --max-cases 1 \
    --reset


In [ ]:
# -------------------------------------------------------------
# 🧪 Step 8.1b: Smoke Test - MedRAX Agent Mode on 1 Case (~45 sec)
# -------------------------------------------------------------
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py \
    --mode medrax \
    --model gemini-3.5-flash-lite \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/smoke_test_medrax" \
    --max-cases 1 \
    --reset


In [ ]:
# -------------------------------------------------------------
# 📊 Option A: Multimodal Baseline Evaluation (Safe Paced + Auto-Resume)
# -------------------------------------------------------------
# Saves every 10 questions to Drive + immediate save on daily quota stop.
# Safe to re-run after switching accounts (NO --reset flag, so it auto-resumes!).
# Optional: Add --max-cases N if you only want to evaluate N cases.
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py \
    --mode baseline \
    --model gemini-3.5-flash-lite \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/gemini_baseline_full" 


In [ ]:
# -------------------------------------------------------------
# 🏥 Option B: MedRAX Agent Evaluation (Safe Paced + Auto-Resume)
# -------------------------------------------------------------
# Coordinates local GPU tools (CheXagent-2-3B, TorchXRayVision) + Gemini 3.5 Flash-Lite.
# Saves every 10 questions to Drive + immediate save on daily quota stop.
# Safe to re-run after switching accounts (NO --reset flag, so it auto-resumes!).
# Optional: Add --max-cases N if you only want to evaluate N cases.
%cd /content/MedRAX-ChestAgentBench

!python scripts/run_evaluation.py \
    --mode medrax \
    --model gemini-3.5-flash-lite \
    --data-dir "$BENCHMARK_DATA_DIR" \
    --output-dir "$MEDRAX_RESULTS_DIR/gemini_medrax_full" 


## 9. 📊 Comparative Analytics: Baseline vs MedRAX Agent
Generate side-by-side comparison tables and accuracy breakdown charts across the 7 clinical axes:
* **Detection**, **Classification**, **Localization**, **Comparison**, **Relationship**, **Diagnosis**, **Characterization**


In [ ]:
# -------------------------------------------------------------
# ⚖️ Compare Baseline vs MedRAX Agent Performance
# -------------------------------------------------------------
%cd /content/MedRAX-ChestAgentBench

!python experiments/compare_runs.py \
    "$MEDRAX_RESULTS_DIR/gemini_baseline_full/predictions.jsonl" \
    "$MEDRAX_RESULTS_DIR/gemini_medrax_full/predictions.jsonl" \
    --output-dir "$MEDRAX_RESULTS_DIR/comparisons"

# Display generated comparison plot
from IPython.display import Image, display
chart_path = Path(os.environ["MEDRAX_RESULTS_DIR"]) / "comparisons" / "accuracy_breakdown.png"
if chart_path.exists():
    display(Image(str(chart_path)))


## 10. 🔄 Synchronize Results & Updates to GitHub
Push any notebook changes or evaluation summaries back to your GitHub repository.


In [ ]:
# Configure Git credentials in Colab
!git config --global user.name "Uy Huynh"
!git config --global user.email "huynhtanuy123@gmail.com"

# Check Git status
%cd /content/MedRAX-ChestAgentBench
!git status -s
